# Tarea

1. Cargar los conjuntos
2. Sobre el conjunto de de entrenamiento ejecutar un metodo de selección de caracteristicas
3. Sobre el subconjunto seleccionado ejecutar varios métodos de modelación.
4. Validar en el conjunto de entrenamiento
5. Validar en el conjunto de tuning
6. Evaluar generalización en el conjunto externo.




In [1]:


import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import make_scorer, matthews_corrcoef, confusion_matrix, accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_validate


## Loading data

In [2]:
# Load the datasets
train_df = pd.read_csv('/content/drive/MyDrive/diplomado_cm_ia/modelacion_test/00-train.csv')
tuning_df = pd.read_csv('/content/drive/MyDrive/diplomado_cm_ia/modelacion_test/01-tuning.csv')
external_df = pd.read_csv('/content/drive/MyDrive/diplomado_cm_ia/modelacion_test/02-external.csv')

# Separate features and target variable
X_train = train_df.drop('Class', axis=1)
y_train = train_df['Class']

X_tuning = tuning_df.drop('Class', axis=1)
y_tuning = tuning_df['Class']

X_external = external_df.drop('Class', axis=1)
y_external = external_df['Class']


## Implementación manual de sensibilidad y especificidad

In [3]:
def sensitivity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tp / (tp + fn)

def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

## seleccion de filtro basada en informacion mutua

In [4]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

def seleccion_kbest(x_train, y_train):

    k = int(x_train.shape[1]/5)

    selector = SelectKBest(
        score_func=mutual_info_classif,
        k=k
    )

    selector.fit(x_train, y_train)

    return selector

## Selección de Características con RandomForest


In [ ]:
def seleccion(x_train, y_train):

    rf_model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    num_variables = int(x_train.shape[1]/5)

    scorer = make_scorer(matthews_corrcoef)

    sfs = SequentialFeatureSelector(
        rf_model,
        n_features_to_select=num_variables,
        direction='backward',
        scoring=scorer,
        cv=10,
        n_jobs=-1
    )

    sfs.fit(x_train, y_train)

    return sfs

## Modelación CV y scoring

In [5]:

scoring = {
    'MCC': make_scorer(matthews_corrcoef),
    'ACC': make_scorer(accuracy_score),
    'SEN': make_scorer(sensitivity),
    'SPE': make_scorer(specificity)
}

def build_model_cv(model, x, y):

  sk_folds = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

  results = cross_validate(
        model,
        x,
        y,
        cv=sk_folds,
        scoring=scoring,
        n_jobs=-1
    )

  model.fit(x, y)

  return model, {
        "MCC_mean": np.mean(results['test_MCC']),
        "MCC_std":  np.std(results['test_MCC']),
        "ACC_mean": np.mean(results['test_ACC']),
        "ACC_std":  np.std(results['test_ACC']),
        "SEN_mean": np.mean(results['test_SEN']),
        "SEN_std":  np.std(results['test_SEN']),
        "SPE_mean": np.mean(results['test_SPE']),
        "SPE_std":  np.std(results['test_SPE'])
    }

## Build model and get scrore

In [6]:
def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    acc = accuracy_score(y_true, y_pred)
    sen = sensitivity(y_true, y_pred)
    spe = specificity(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)

    return {
        "ACC": acc,
        "SEN": sen,
        "SPE": spe,
        "MCC": mcc
    }


def build_model_and_score(model_name, model, x_train_red, y_tune_red, x_ext_red):
  model, train_cv_scores = build_model_cv(model, x_train_reduced, y_train)

  tune_predict = model.predict(x_tuning_reduced)
  ext_predict = model.predict(x_external_reduced)

  tune_scores = compute_metrics(y_tuning, tune_predict)
  ext_scores = compute_metrics(y_external, ext_predict)

  return {
        "model": model_name,

        "train": {
            "ACC": train_cv_scores["ACC_mean"],
            "SEN": train_cv_scores["SEN_mean"],
            "SPE": train_cv_scores["SPE_mean"],
            "MCC": train_cv_scores["MCC_mean"]
        },

        "tune": tune_scores,

        "external": ext_scores
    }




## Build matrix metric scoring

In [7]:
def result_to_table(results_list):
  rows = []

  for r in results_list:
    row = {
        "Model" : r["model"],

        "Train_ACC" : r["train"]["ACC"],
        "Train_SEN" : r["train"]["SEN"],
        "Train_SPE" : r["train"]["SPE"],
        "Train_MCC" : r["train"]["MCC"],

        "Tune_ACC"  : r["tune"]["ACC"],
        "Tune_SEN"  : r["tune"]["SEN"],
        "Tune_SPE"  : r["tune"]["SPE"],
        "Tune_MCC"  : r["tune"]["MCC"],

        "Ext_ACC"   : r["external"]["ACC"],
        "Ext_SEN"   : r["external"]["SEN"],
        "Ext_SPE"   : r["external"]["SPE"],
        "Ext_MCC"   : r["external"]["MCC"]
    }

    rows.append(row)

  final_table = pd.DataFrame(rows)
  return final_table


## Ejecutar proceso

In [ ]:
sfs = seleccion_kbest(X_train, y_train)

x_train_reduced = sfs.transform(X_train)
x_tuning_reduced  = sfs.transform(X_tuning)
x_external_reduced  = sfs.transform(X_external)

models = {
    "svc_poly": SVC(kernel='poly'),
    "svc_rbf": SVC(kernel='rbf'),
    "svc_sigmoid": SVC(kernel='sigmoid'),
    "knn_3" : KNeighborsClassifier(n_neighbors=3),
    "knn_5" : KNeighborsClassifier(n_neighbors=5),
    "knn_7" : KNeighborsClassifier(n_neighbors=7),
    "knn_9" : KNeighborsClassifier(n_neighbors=9)

}

results = []

for model_name,model  in models.items():
  results.append(build_model_and_score(model_name, model, x_train_reduced, x_tuning_reduced, x_external_reduced))

scores =result_to_table(results)

display(scores)

,Model,Train_ACC,Train_SEN,Train_SPE,Train_MCC,Tune_ACC,Tune_SEN,Tune_SPE,Tune_MCC,Ext_ACC,Ext_SEN,Ext_SPE,Ext_MCC
0,svc_poly,0.777632,0.715556,0.837778,0.573600,0.790323,0.806452,0.774194,0.580948,0.687981,0.684523,0.778589,0.184967
1,svc_rbf,0.702368,0.826667,0.577778,0.427288,0.709677,0.838710,0.580645,0.434057,0.830531,0.841519,0.542579,0.191247
2,svc_sigmoid,0.394737,0.610000,0.190000,-0.232995,0.435484,0.709677,0.161290,-0.154303,0.831694,0.861573,0.048662,-0.049407
3,knn_3,0.807632,0.786667,0.826667,0.621297,0.774194,0.774194,0.774194,0.548387,0.709086,0.708662,0.720195,0.174931
4,knn_5,0.827632,0.826667,0.826667,0.668394,0.774194,0.774194,0.774194,0.548387,0.717403,0.716089,0.751825,0.191924
5,knn_7,0.832632,0.836667,0.826667,0.678510,0.774194,0.806452,0.741935,0.549532,0.714094,0.713119,0.739659,0.185296
6,knn_9,0.817632,0.806667,0.826667,0.652799,0.774194,0.806452,0.741935,0.549532,0.717224,0.716461,0.737226,0.186246
